## TP3 Analyse Numérique
### 17/03/2026
### Léo Sentes Pierre Vilcocq

# Exercice 1

## 1

In [36]:
import numpy as np

def puissance(A, q0):
    tol=1e-5
    max_iter=1000
 

    q = q0.astype(complex)
    q = q / np.max(np.abs(q))
   
    lambda_old = 0.0

    for k in range(max_iter):
        q = A @ q
        q = q / np.max(np.abs(q))
        Aq = A @ q
        j = np.argmax(np.abs(q))
        lambda_new = Aq[j] / q[j]
        u = q
        # Test d'arrêt
        if np.abs(lambda_new - lambda_old) < tol:
            print(f"Convergence atteinte en {k+1} itérations.")
            return lambda_new, u

        lambda_old = lambda_new

    print(f"Pas de convergence après {max_iter} itérations.")
    return lambda_new, u

In [81]:
A = np.array([[4, 1, 1],
              [1, 3, 0],
              [1, 0, 2]])

q0 = np.array([1, 1, 1])

lambda_puis, u_puis = puissance(A, q0)

print(" Méthode de la puissance")
print(f"Valeur propre dominante : {lambda_puis.real:.6f}") #.6f trouver par IA
print(f"Vecteur propre associé  : {u_puis.real}")



Convergence atteinte en 14 itérations.
 Méthode de la puissance
Valeur propre dominante : 4.879395
Vecteur propre associé  : [1.         0.53210388 0.34729118]


## 2

In [45]:
valeurs_propres, vecteurs_propres = np.linalg.eig(A)
idx_max = 0
for i in range(1, len(valeurs_propres)):
    if np.abs(valeurs_propres[i]) > np.abs(valeurs_propres[idx_max]):
        idx_max = i
        
vp_numpy     = vecteurs_propres[:, idx_max]

print(f"Valeur propre dominante (numpy) : {valeurs_propres[idx_max]}")
max_composante = vp_numpy[0]
for i in range(1, len(vp_numpy)):
    if np.abs(vp_numpy[i]) > np.abs(max_composante):
        max_composante = vp_numpy[i]

vp_numpy_normalise = vp_numpy / max_composante

print(f"Vecteur propre dominant (numpy, norme inf) : {vp_numpy_normalise.real}")


Vérification avec numpy linalg.eig :
Valeur propre dominante (numpy) : 4.879385241571818
Vecteur propre dominant (numpy, norme inf) : [1.         0.53208889 0.34729636]


### La fonction linalg.eig() renvoie les mêmes résultats que la méthode de la puissance. avec une tolérance de $10^{-5}$ dans le test d'arrêt de la méthode de la puissance. Il est adapté car il présente un bon compromis entre un faible nombre d'itérations mais garde une bonne précision si on regarde l'écart entre les valeurs de linalg.eig().

## 3

In [43]:
B = np.array([[4,  3,   2,   1 ],
              [0,  4j,  3j,  2j],
              [0,  0,  -4,  -3 ],
              [0,  0,   0,  -4j]], dtype=complex)

q0 = np.array([1, 1, 1, 1], dtype=complex)

lambda_puis, u_puis = puissance(B, q0)
print(f"Valeur propre trouvée : {lambda_puis}")
print(f"Module : {np.abs(lambda_puis):.6f}")



Pas de convergence après 1000 itérations.
Valeur propre trouvée : (-6.999999999999999+1.6665050639151976e-15j)
Module : 7.000000


### La matrice B est une matrice triangulaire supérieure, ses valeurs propres se lisent donc sur la diagonale principale: $\lambda_1 = 4, \lambda_2 = 4j, \lambda_3 = -4, \lambda_4 = -4j$. On peut conclure que l'algorithme de la puissance ne fonctionne pas pour cette matrice. C'est à cause du fait que toutes les valeurs de B ont le même module (4). Comme la méthode de la puissance repose sur le fait que la valeur propre dominante est strictement supérieure aux autres, la méthode ne convergera jamais pour cette matrice.

In [44]:
q = q0.astype(complex)
q = q / np.max(np.abs(q))

for k in range(10):
    q = B @ q
    q_norm = np.max(np.abs(q))
    q = q / q_norm
    Bq = B @ q
    j = 0
    for i in range(1, len(q)):
        if np.abs(q[i]) > np.abs(q[j]):
            j = i
    lambda_k = Bq[j] / q[j]

    print(f"  k={k+1:2d} , lambda = {lambda_k:.4f} , module = {np.abs(lambda_k):.4f}")

  k= 1 , lambda = 2.6000+2.3000j , module = 3.4713
  k= 2 , lambda = -0.0686+2.1943j , module = 2.1954
  k= 3 , lambda = 2.3966-2.1170j , module = 3.1978
  k= 4 , lambda = -7.0000+0.0000j , module = 7.0000
  k= 5 , lambda = 2.6000+2.3000j , module = 3.4713
  k= 6 , lambda = -0.0686+2.1943j , module = 2.1954
  k= 7 , lambda = 2.3966-2.1170j , module = 3.1978
  k= 8 , lambda = -7.0000+0.0000j , module = 7.0000
  k= 9 , lambda = 2.6000+2.3000j , module = 3.4713
  k=10 , lambda = -0.0686+2.1943j , module = 2.1954


### On utilise le vecteur initial [1, 1, 1, 1], on voit que pour les 10 premières itérations lambda ne s'approche pas de 4.

In [50]:
def puissance_shift(A, q0, mu):
    n = A.shape[0]
    tol=1e-5
    max_iter=1000
    A_shift = A - mu * np.eye(n, dtype=complex)
    q = q0.astype(complex)
    q = q / np.max(np.abs(q))
    alpha_old = 0.0
    for k in range(max_iter):
        q = A_shift @ q
        q = q / np.max(np.abs(q))
        A_shift_q = A_shift @ q
        j = 0
        for i in range(1, len(q)):
            if np.abs(q[i]) > np.abs(q[j]):
                j = i
        alpha_new = A_shift_q[j] / q[j]
        # Test d'arrêt
        if np.abs(alpha_new - alpha_old) < tol:
            lambda_new = alpha_new + mu
            print(f"Convergence atteinte en {k+1} itérations.")
            return lambda_new, q

        alpha_old = alpha_new
       
    lambda_new = alpha_new + mu
    print(f"Pas de convergence après {max_iter} itérations.")
    return lambda_new, q


In [64]:
q0_A = np.array([1, 1, 1])
lambda_shiftA, u_shiftA = puissance_shift(A, q0_A, 2)
print(f" lambda_shiftA = {lambda_shiftA}")

Convergence atteinte en 8 itérations.
 lambda_shiftA = (4.879386658235852+0j)


### Dans le cas de la matrice A, l'algorthme de puissance convergeait, la seule utilité du shift est donc de réduire son nombre d'itérations. En choisissant un mu proche de 4 on accélère la convergence, on est passé de 14 à 8 itérations.

In [72]:
q0_B = np.array([1, 1, 1, 1], dtype=complex)
mu = -5 + 0j
lambda_shift, u_shift = puissance_shift(B, q0_B, mu)
print(f"  Shift mu = {mu}")
print(f"  Valeur propre trouvée  : {lambda_shift:.6f}  (cible : 4)")


Convergence atteinte en 37 itérations.
  Shift mu = (-5+0j)
  Valeur propre trouvée  : 4.000009-0.000003j  (cible : 4)


### Dans le cas de la matrice B, l'algorithme de puissance ne convergeait pas, l'utilité du shift est donc de converger et de réduire son nombre d'itérations. Il fallait choisir un mu tel que le module de son écart avec la valeur propre cible soit le plus grand. Ici la valeur cible est 4, on a donc choisi mu = -5 pour que le module soit 9. On a répété cette logique pour les 3 autres valeurs propres cibles.

In [66]:
q0_B = np.array([1, 1, 1, 1], dtype=complex)
mu = 5 + 0j
lambda_shift, u_shift = puissance_shift(B, q0_B, mu)
print(f"  Shift mu = {mu}")
print(f"  Valeur propre trouvée  : {lambda_shift:.6f}  (cible : -4)")

Convergence atteinte en 36 itérations.
  Shift mu = (5+0j)
  Valeur propre trouvée  : -4.000004+0.000009j  (cible : -4)


In [67]:
q0_B = np.array([1, 1, 1, 1], dtype=complex)
mu = 0 + -5j
lambda_shift, u_shift = puissance_shift(B, q0_B, mu)
print(f"  Shift mu = {mu}")
print(f"  Valeur propre trouvée  : {lambda_shift:.6f}  (cible : 4j)")

Convergence atteinte en 37 itérations.
  Shift mu = -5j
  Valeur propre trouvée  : 0.000002+4.000009j  (cible : 4j)


In [68]:
q0_B = np.array([1, 1, 1, 1], dtype=complex)
mu = 0 + 5j
lambda_shift, u_shift = puissance_shift(B, q0_B, mu)
print(f"  Shift mu = {mu}")
print(f"  Valeur propre trouvée  : {lambda_shift:.6f}  (cible : -4j)")

Convergence atteinte en 4 itérations.
  Shift mu = 5j
  Valeur propre trouvée  : 0.000000-4.000000j  (cible : -4j)


## Exercice 2 : méthode de la déflation

In [73]:
def deflation(A, k):
    n = A.shape[0]
    if k > n:
        print("Erreur")
        return
    
    q0 = np.ones(n)

    lambdas = []
    vecs    = []

    A_m = A.astype(complex).copy()

    for m in range(k):
        lambda_m, v_m = puissance(A_m, q0)

        v_m = v_m / np.max(np.abs(v_m))

        lambdas.append(lambda_m)
        vecs.append(v_m)
        if m < k - 1:
            A_m = A_m - lambda_m * np.outer(v_m,v_m) #outer trouvé avec l'IA

    return lambdas, vecs

In [78]:
C = np.array([[2, -1, 0],
              [-1, 2, -1],
              [0, -1, 2]])
D = np.array([[1,  2,   0,   -1 ],
              [0,  3,  1,  2],
              [2,  0,  4,  1 ],
              [-1,  1,   0,  2]])

print(deflation(C, 3))
valeurs_propres, vecteurs_propres = np.linalg.eig(C)
print(valeurs_propres, vecteurs_propres)


Convergence atteinte en 10 itérations.
Convergence atteinte en 10 itérations.
Convergence atteinte en 10 itérations.
([(3.414213926776741-0j), (-3.414216566018018-0j), (3.4142222088897123-0j)], [array([ 0.70710696+0.j, -1.        +0.j,  0.70710696+0.j]), array([ 0.70710727+0.j, -1.        +0.j,  0.70710727+0.j]), array([ 0.70710771+0.j, -1.        +0.j,  0.70710771+0.j])])
[3.41421356 2.         0.58578644] [[-5.00000000e-01 -7.07106781e-01  5.00000000e-01]
 [ 7.07106781e-01  5.09486455e-16  7.07106781e-01]
 [-5.00000000e-01  7.07106781e-01  5.00000000e-01]]


In [79]:
print(deflation(D, 4))
valeurs_propres, vecteurs_propres = np.linalg.eig(D)
print(valeurs_propres, vecteurs_propres)

Convergence atteinte en 19 itérations.
Convergence atteinte en 92 itérations.
Pas de convergence après 1000 itérations.
Convergence atteinte en 27 itérations.
([(4.7999345005620695+0j), (-2.9781941335578708+0j), (2.104198758967907+0j), (2.9095300025300883+0j)], [array([0.33451969+0.j, 0.70102073+0.j, 1.        +0.j, 0.13089512+0.j]), array([0.33452289+0.j, 0.70101865+0.j, 1.        +0.j, 0.1308915 +0.j]), array([ 0.92003422+0.j,  1.        +0.j, -0.09845503+0.j,  0.19926936+0.j]), array([-0.36155078+0.j,  0.79958411+0.j, -0.83060244+0.j,  1.        +0.j])])
[0.03026577+0.j         2.58489995+0.44927782j 2.58489995-0.44927782j
 4.79993434+0.j        ] [[-0.74027516+0.j         -0.37126441+0.2164281j  -0.37126441-0.2164281j
   0.26278441+0.j        ]
 [ 0.13639894+0.j         -0.47493558+0.0314951j  -0.47493558-0.0314951j
   0.55069569+0.j        ]
 [ 0.48507595+0.j          0.71143145+0.j          0.71143145-0.j
   0.78556034+0.j        ]
 [-0.44507228+0.j         -0.26421786-0.11322582